In [1]:
%env WORKDIR=/tmp/vault

env: WORKDIR=/tmp/vault


In [2]:
! pip install dotenv

In [3]:
import os
from dotenv import load_dotenv

load_dotenv("/tmp/vault/config.env")

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')

# Basic Configuration

In [4]:
%%bash

vault secrets disable transit


vault secrets enable transit
# Create a new key named "kms" in the transit secrets engine
# AES-GCM is the default encryption algorithm used by the transit secrets engine. It is a symmetric encryption algorithm that provides both confidentiality and integrity for the data being encrypted. AES-GCM is widely used in modern cryptography and is considered secure for most applications.

vault write -f transit/keys/kms

# Rotate key every 1 month (30 days) to ensure that the key is regularly updated and to reduce the risk of compromise. 
# By default will will be able to use old keys for decryption, but new encryption operations will use the latest key version. 
# This helps to ensure that sensitive data is protected with the most up-to-date cryptographic algorithms and reduces the risk of data breaches.
vault write transit/keys/kms/config auto_rotate_period=30d


Success! Disabled the secrets engine (if it existed) at: transit/
Success! Enabled the transit secrets engine at: transit/
Key                       Value
---                       -----
allow_plaintext_backup    false
auto_rotate_period        0s
deletion_allowed          false
derived                   false
exportable                false
imported_key              false
keys                      map[1:1789739161]
latest_version            1
min_available_version     0
min_decryption_version    1
min_encryption_version    0
name                      kms
supports_decryption       true
supports_derivation       true
supports_encryption       true
supports_signing          false
type                      aes256-gcm96
Key                       Value
---                       -----
allow_plaintext_backup    false
auto_rotate_period        720h
deletion_allowed          false
derived                   false
exportable                false
imported_key              false
keys               

# Encrypt and Decrypt with CLI

In [5]:
%%bash
# Encrypt a sample plaintext using the "kms" key in the transit secrets engine
PLAINTEXT=$(echo -n "Hello, Vault!" | base64)
CIPHERTEXT=$(vault write transit/encrypt/kms plaintext=$PLAINTEXT -format=json | jq -r '.data.ciphertext')
echo "Ciphertext: $CIPHERTEXT"

# Decrypt the ciphertext using the "kms" key in the transit secrets engine
DECRYPTED=$(vault write transit/decrypt/kms ciphertext=$CIPHERTEXT -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted: $DECRYPTED"   


Ciphertext: vault:v1:J6e1Ym7wplwRwvGEYrT556+JYqqrZhSXJeIwU+3ryU/eupZ0UmII6RE=
Decrypted: Hello, Vault!


# Encrypt and Decrypt with curl

In [6]:
%%bash
# Same exercise as above but using curl insetad of the vault cli
# Encrypt a sample plaintext using the "kms" key in the transit secrets engine
PLAINTEXT=$(echo -n "Hello, Vault!" | base64)
curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data '{"plaintext": "'"$PLAINTEXT"'"}' \
     $VAULT_ADDR/v1/transit/encrypt/kms | jq -r '.data.ciphertext'  > ciphertext.txt
CIPHERTEXT=$(cat ciphertext.txt)
echo "Ciphertext: $CIPHERTEXT"

# Decrypt the ciphertext using the "kms" key in the transit secrets engine
DECRYPTED=$(curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data '{"ciphertext": "'"$CIPHERTEXT"'"}' \
     $VAULT_ADDR/v1/transit/decrypt/kms | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted: $DECRYPTED"


Ciphertext: vault:v1:FxdwhkNgzGuZqO22fv9SbreviZlirl2N8BsSHZLAy0ooR8E2OoVD96Y=
Decrypted: Hello, Vault!


# Batch operations

In [7]:
%%bash
# Batch - Generate 1000 inputs for batch encryption

echo "Generating batch_input.json with 1000 unique card entries..."

# Start building JSON array
echo '{' > batch_input.json
echo '"batch_input": [' >> batch_input.json

# Generate 1000 entries, each with a unique card number derived from the index
for i in {1..1000}; do
    # Build a unique 16-digit card number: Visa prefix (4) + zero-padded index
    num=$(printf "4%015d" $i)
    card_number="${num:0:4} ${num:4:4} ${num:8:4} ${num:12:4}"

    # Base64 encode the plaintext
    encoded_text=$(echo -n "$card_number" | base64)

    # Add JSON entry (no trailing comma on the last entry)
    if [ $i -lt 1000 ]; then
        echo "    {\"plaintext\": \"$encoded_text\"}," >> batch_input.json
    else
        echo "    {\"plaintext\": \"$encoded_text\"}" >> batch_input.json
    fi

    # Progress indicator
    if [ $((i % 100)) -eq 0 ]; then
        echo "Generated $i entries..."
    fi
done

# Close JSON array
echo ']' >> batch_input.json
echo '}' >> batch_input.json

echo "Generated batch_input.json with 1000 unique entries"
echo "File size: $(wc -c < batch_input.json) bytes"
echo "Total lines: $(wc -l < batch_input.json) lines"

echo -e "\nFirst 10 lines of the file:"
head -10 batch_input.json

echo -e "\nLast 10 lines of the file:"
tail -10 batch_input.json


Generating batch_input.json with 1000 unique card entries...
Generated 100 entries...
Generated 200 entries...
Generated 300 entries...
Generated 400 entries...
Generated 500 entries...
Generated 600 entries...
Generated 700 entries...
Generated 800 entries...
Generated 900 entries...
Generated 1000 entries...
Generated batch_input.json with 1000 unique entries
File size:    51022 bytes
Total lines:     1004 lines

First 10 lines of the file:
{
"batch_input": [
    {"plaintext": "NDAwMCAwMDAwIDAwMDAgMDAwMQ=="},
    {"plaintext": "NDAwMCAwMDAwIDAwMDAgMDAwMg=="},
    {"plaintext": "NDAwMCAwMDAwIDAwMDAgMDAwMw=="},
    {"plaintext": "NDAwMCAwMDAwIDAwMDAgMDAwNA=="},
    {"plaintext": "NDAwMCAwMDAwIDAwMDAgMDAwNQ=="},
    {"plaintext": "NDAwMCAwMDAwIDAwMDAgMDAwNg=="},
    {"plaintext": "NDAwMCAwMDAwIDAwMDAgMDAwNw=="},
    {"plaintext": "NDAwMCAwMDAwIDAwMDAgMDAwOA=="},

Last 10 lines of the file:
    {"plaintext": "NDAwMCAwMDAwIDAwMDAgMDk5Mw=="},
    {"plaintext": "NDAwMCAwMDAwIDAwMDAgMDk5NA==

## Encrypt

In [8]:
%%bash
# Execute batch encryption with 1000 inputs

echo "Executing batch encryption of 1000 entries..."
echo "Start time: $(date)"

# Execute the batch operation and measure time
start_time=$(date +%s%N)

curl -k -s \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request POST \
    --data @batch_input.json \
    $VAULT_ADDR/v1/transit/encrypt/orders > batch_output.json

end_time=$(date +%s%N)
duration_ns=$((end_time - start_time))
duration_seconds=$(echo "scale=6; $duration_ns / 1000000000" | bc)

echo "Batch encryption completed!"
echo "Duration: ${duration_ns} nanoseconds"
echo "Duration: ${duration_seconds} seconds"

# Calculate operations per second (avoid division by zero)
if [ $duration_ns -gt 0 ]; then
    # Use bc for floating point arithmetic: (1000 * 1000000000) / duration_ns
    ops_per_second=$(echo "scale=2; 1000000000000 / $duration_ns" | bc)
    echo "Processing rate: ${ops_per_second} operations/second"
else
    echo "Processing rate: Unable to calculate (duration too small)"
fi

echo -e "\nOutput file size: $(wc -c < batch_output.json) bytes"

# Check if the operation was successful
if jq -e '.data.batch_results' batch_output.json > /dev/null 2>&1; then
    result_count=$(jq '.data.batch_results | length' batch_output.json)
    echo "Successfully encrypted $result_count entries"
    
    echo -e "\nFirst encrypted result:"
    jq '.data.batch_results[0]' batch_output.json
    
    echo -e "\nLast encrypted result:"
    jq '.data.batch_results[-1]' batch_output.json
else
    echo "Batch operation failed. Response:"
    jq '.' batch_output.json
fi


Executing batch encryption of 1000 entries...
Start time: Fri Sep 18 15:46:07 CEST 2026
Batch encryption completed!
Duration: 97276000 nanoseconds
Duration: .097276 seconds
Processing rate: 10280.02 operations/second

Output file size:   122200 bytes
Successfully encrypted 1000 entries

First encrypted result:
{
  "ciphertext": "vault:v1:EUYt+XfgyKaby/yRreI+WfRgGgeQGvhFy9c3GWdmzi0nhbFKmqyXL5eX3+K2UPU=",
  "key_version": 1,
  "reference": ""
}

Last encrypted result:
{
  "ciphertext": "vault:v1:dR2d6e9du924ShmlhRm1H0qQu3tnp5KvW7sj0gpW4VeQ/HzENN1oRK84C8P7Low=",
  "key_version": 1,
  "reference": ""
}


## Decrypt

### Adapt input batch file for decrypt operation

In [9]:
%%bash
# Extract encrypted results and format for batch decryption

echo "Processing batch_output.json for decryption..."

# Use jq to build a properly formatted batch_decrypt_input.json directly
jq '{batch_input: [.data.batch_results[] | {ciphertext: .ciphertext}]}' \
    batch_output.json > batch_decrypt_input.json

echo "Created batch_decrypt_input.json for decryption"
echo "File size: $(wc -c < batch_decrypt_input.json) bytes"

echo -e "\nFirst 5 lines of decrypt input:"
head -5 batch_decrypt_input.json

echo -e "\nValidating JSON format:"
if jq empty batch_decrypt_input.json 2>/dev/null; then
    echo "JSON format is valid"
    entry_count=$(jq '.batch_input | length' batch_decrypt_input.json)
    echo "Total entries to decrypt: $entry_count"
else
    echo "JSON format is invalid"
fi


Processing batch_output.json for decryption...
Created batch_decrypt_input.json for decryption
File size:   109026 bytes

First 5 lines of decrypt input:
{
  "batch_input": [
    {
      "ciphertext": "vault:v1:EUYt+XfgyKaby/yRreI+WfRgGgeQGvhFy9c3GWdmzi0nhbFKmqyXL5eX3+K2UPU="
    },

Validating JSON format:
JSON format is valid
Total entries to decrypt: 1000


### Decrypt

In [10]:
%%bash
# Execute batch decryption with 1000 entries

echo "Executing batch decryption of 1000 entries..."
echo "Start time: $(date)"

# Execute the batch decryption operation and measure time
start_time=$(date +%s%N)

curl -k -s \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request POST \
    --data @batch_decrypt_input.json \
    $VAULT_ADDR/v1/transit/decrypt/orders > batch_decrypt_output.json

end_time=$(date +%s%N)
duration_ns=$((end_time - start_time))
duration_seconds=$(echo "scale=6; $duration_ns / 1000000000" | bc)

echo "Batch decryption completed!"
echo "Duration: ${duration_ns} nanoseconds"
echo "Duration: ${duration_seconds} seconds"

# Calculate operations per second (avoid division by zero)
if [ $duration_ns -gt 0 ]; then
    # Use bc for floating point arithmetic: (1000 * 1000000000) / duration_ns
    ops_per_second=$(echo "scale=2; 1000000000000 / $duration_ns" | bc)
    echo "Processing rate: ${ops_per_second} operations/second"
else
    echo "Processing rate: Unable to calculate (duration too small)"
fi

echo -e "\nOutput file size: $(wc -c < batch_decrypt_output.json) bytes"

# Check if the operation was successful
if jq -e '.data.batch_results' batch_decrypt_output.json > /dev/null 2>&1; then
    result_count=$(jq '.data.batch_results | length' batch_decrypt_output.json)
    echo "Successfully decrypted $result_count entries"
    
    echo -e "\nFirst decrypted result (base64 encoded):"
    jq '.data.batch_results[0]' batch_decrypt_output.json
    
    echo -e "\nFirst decrypted result (decoded):"
    jq -r '.data.batch_results[0].plaintext' batch_decrypt_output.json | base64 -d
    
    echo -e "\nLast decrypted result (decoded):"
    jq -r '.data.batch_results[-1].plaintext' batch_decrypt_output.json | base64 -d
    
    # Show a few more samples
    echo -e "\nSample of decrypted entries (first 5):"
    for i in {0..4}; do
        decrypted=$(jq -r ".data.batch_results[$i].plaintext" batch_decrypt_output.json | base64 -d)
        echo "Entry $((i+1)): $decrypted"
    done
else
    echo "Batch decryption failed. Response:"
    jq '.' batch_decrypt_output.json
fi

Executing batch decryption of 1000 entries...
Start time: Fri Sep 18 15:46:07 CEST 2026
Batch decryption completed!
Duration: 82679000 nanoseconds
Duration: .082679 seconds
Processing rate: 12094.96 operations/second

Output file size:    60200 bytes
Successfully decrypted 1000 entries

First decrypted result (base64 encoded):
{
  "plaintext": "NDAwMCAwMDAwIDAwMDAgMDAwMQ==",
  "reference": ""
}

First decrypted result (decoded):
4000 0000 0000 0001
Last decrypted result (decoded):
4000 0000 0000 1000
Sample of decrypted entries (first 5):
Entry 1: 4000 0000 0000 0001
Entry 2: 4000 0000 0000 0002
Entry 3: 4000 0000 0000 0003
Entry 4: 4000 0000 0000 0004
Entry 5: 4000 0000 0000 0005


# Rotate key

In [11]:
%%bash
# Rotate the key to demonstrate decryption of old ciphertexts with a new key version
echo "Rotating the 'kms' key to create a new version..."    

vault write -f transit/keys/kms/rotate 

Rotating the 'kms' key to create a new version...
Key                       Value
---                       -----
allow_plaintext_backup    false
auto_rotate_period        720h
deletion_allowed          false
derived                   false
exportable                false
imported_key              false
keys                      map[1:1789739161 2:1789739168]
latest_version            2
min_available_version     0
min_decryption_version    1
min_encryption_version    0
name                      kms
supports_decryption       true
supports_derivation       true
supports_encryption       true
supports_signing          false
type                      aes256-gcm96


In [12]:
%%bash
# Encrypt a sample plaintext using the "kms" key in the transit secrets engine
PLAINTEXT=$(echo -n "Hello, Vault!" | base64)
CIPHERTEXT=$(vault write transit/encrypt/kms plaintext=$PLAINTEXT -format=json | jq -r '.data.ciphertext')
echo "Ciphertext: $CIPHERTEXT"

Ciphertext: vault:v2:KeISs3+QTOmEgpxenbgLwoErIxlu1Jmi5+OT7KVtVnROBS5lVJWrkp4=


## Decrypt data with old and new keys

In [13]:
%%bash
# Encrypt using the new key version to demonstrate that it can still be decrypted after rotation
PLAINTEXT=$(echo -n "Hello, Vault! (New Key Version)" | base64)
CIPHERTEXT_NEW=$(vault write transit/encrypt/kms plaintext=$PLAINTEXT -format=json | jq -r '.data.ciphertext')
echo "Ciphertext (New Key Version): $CIPHERTEXT_NEW"

# Decrypt the new ciphertext using the "kms" key in the transit secrets engine
DECRYPTED_NEW=$(vault write transit/decrypt/kms ciphertext=$CIPHERTEXT_NEW -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted (New Key Version): $DECRYPTED_NEW" 


Ciphertext (New Key Version): vault:v2:UWmn4GTR62bhIL+hUKbopaDSXHjtxsWeMpaIyNW13jHx2U1yxT94MAviTa7D+Wz+uc5XgyMG507gxF8=
Decrypted (New Key Version): Hello, Vault! (New Key Version)


In [14]:
%%bash
CYPHERTEXT_OLD=$(cat ciphertext.txt)
echo "Old Ciphertext: $CYPHERTEXT_OLD"
# Decrypt the old ciphertext using the "kms" key in the transit secrets engine
DECRYPTED_OLD=$(vault write transit/decrypt/kms ciphertext=$CYPHERTEXT_OLD -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted (Old Key Version): $DECRYPTED_OLD"

Old Ciphertext: vault:v1:FxdwhkNgzGuZqO22fv9SbreviZlirl2N8BsSHZLAy0ooR8E2OoVD96Y=
Decrypted (Old Key Version): Hello, Vault!


# Rewrap

In [15]:
%%bash
# Rewrap the old ciphertext to the new key version
CYPHERTEXT_OLD=$(cat ciphertext.txt)
echo "Old Ciphertext: $CYPHERTEXT_OLD"

# Rewrap the old ciphertext to the new key version using the "kms" key in the transit secrets engine
REWRAPPED_CIPHERTEXT=$(vault write transit/rewrap/kms ciphertext=$CYPHERTEXT_OLD -format=json | jq -r '.data.ciphertext')
echo "Rewrapped Ciphertext (New Key Version): $REWRAPPED_CIPHERTEXT"    

# Decrypt the rewrapped ciphertext using the "kms" key in the transit secrets engine
DECRYPTED_REWRAPPED=$(vault write transit/decrypt/kms ciphertext=$REWRAPPED_CIPHERTEXT -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted (Rewrapped to New Key Version): $DECRYPTED_REWRAPPED"

Old Ciphertext: vault:v1:FxdwhkNgzGuZqO22fv9SbreviZlirl2N8BsSHZLAy0ooR8E2OoVD96Y=


Rewrapped Ciphertext (New Key Version): vault:v2:6XbIUOYFFzgvGSlCnHGu9jw3oX3GoJ13RecnNQpmZRBg9jSvRfunKP4=
Decrypted (Rewrapped to New Key Version): Hello, Vault!


# Verifica decoding operations con datos incorrectos
Encrypt three plaintexts in one `batch_input` request, then alter two ciphertexts before a batch decrypt:
1. Wrong key version (`vault:vN:` → `vault:v99:`) so Vault cannot find that key version
2. Flip one character in the ciphertext payload so AES-GCM authentication fails
3. Leave the third ciphertext untouched as a control

Batch decrypt continues even when some items fail: failed entries return `error`, valid ones return `plaintext`.

In [16]:
%%bash
# Encrypt 3 plaintexts in one batch_input request, tamper two ciphertexts, then batch decrypt

P1=$(echo -n "Alice 4111111111111111" | base64)
P2=$(echo -n "Bob 5500000000000004" | base64)
P3=$(echo -n "Carol 340000000000009" | base64)

jq -n --arg p1 "$P1" --arg p2 "$P2" --arg p3 "$P3" \
  '{batch_input: [{plaintext: $p1}, {plaintext: $p2}, {plaintext: $p3}]}' \
  > tamper_encrypt_input.json

echo "=== Batch encrypt input ==="
jq . tamper_encrypt_input.json

curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data @tamper_encrypt_input.json \
     $VAULT_ADDR/v1/transit/encrypt/kms > tamper_encrypt_output.json

echo ""
echo "=== Original ciphertexts ==="
jq -r '.data.batch_results[] | .ciphertext' tamper_encrypt_output.json

C1=$(jq -r '.data.batch_results[0].ciphertext' tamper_encrypt_output.json)
C2=$(jq -r '.data.batch_results[1].ciphertext' tamper_encrypt_output.json)
C3=$(jq -r '.data.batch_results[2].ciphertext' tamper_encrypt_output.json)

# 1) Force a key version that does not exist (vault:vN:payload -> vault:v99:payload)
C1_TAMPERED="vault:v99:$(echo "$C1" | cut -d: -f3-)"

# 2) Flip the first character of the ciphertext payload (keeps vault:vN: prefix)
C2_PREFIX="$(echo "$C2" | cut -d: -f1-2):"
C2_PAYLOAD=$(echo "$C2" | cut -d: -f3-)
C2_FIRST=$(echo "$C2_PAYLOAD" | cut -c1)
C2_REST=$(echo "$C2_PAYLOAD" | cut -c2-)
if [ "$C2_FIRST" = "A" ]; then C2_NEW=B; else C2_NEW=A; fi
C2_TAMPERED="${C2_PREFIX}${C2_NEW}${C2_REST}"

jq -n --arg c1 "$C1_TAMPERED" --arg c2 "$C2_TAMPERED" --arg c3 "$C3" \
  '{batch_input: [{ciphertext: $c1}, {ciphertext: $c2}, {ciphertext: $c3}]}' \
  > tamper_decrypt_input.json

echo ""
echo "=== Tampered batch_input for decrypt ==="
jq . tamper_decrypt_input.json

curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data @tamper_decrypt_input.json \
     $VAULT_ADDR/v1/transit/decrypt/kms > tamper_decrypt_output.json

echo ""
echo "=== Batch decrypt results ==="
jq '.data.batch_results' tamper_decrypt_output.json

echo ""
echo "=== Decoded successful entries ==="
jq -r '.data.batch_results[] | select(.plaintext != null) | .plaintext' tamper_decrypt_output.json \
  | while read -r b64; do
        echo "$b64" | base64 --decode
        echo
    done


=== Batch encrypt input ===
{
  "batch_input": [
    {
      "plaintext": "QWxpY2UgNDExMTExMTExMTExMTExMQ=="
    },
    {
      "plaintext": "Qm9iIDU1MDAwMDAwMDAwMDAwMDQ="
    },
    {
      "plaintext": "Q2Fyb2wgMzQwMDAwMDAwMDAwMDA5"
    }
  ]
}

=== Original ciphertexts ===
vault:v2:LKcAAxRwIEC1JHMbfgxd4wZ3Bojr5JgxS9hahbkw5g0KxQNOitn0G0OhupojLz4ZWqU=
vault:v2:thFlFxmlJbRsuKd0vDl5ZcGDyWyyWd6iHaTBmzCU0TCJgX+27AKzsBOPmf/LNVAd
vault:v2:GJl6vWlY/gOF1Q6quIvRie8EG59SC811iyPuuVRfm6nba/3+/tZ+8j5YfKBOteUd0Q==

=== Tampered batch_input for decrypt ===
{
  "batch_input": [
    {
      "ciphertext": "vault:v99:LKcAAxRwIEC1JHMbfgxd4wZ3Bojr5JgxS9hahbkw5g0KxQNOitn0G0OhupojLz4ZWqU="
    },
    {
      "ciphertext": "vault:v2:AhFlFxmlJbRsuKd0vDl5ZcGDyWyyWd6iHaTBmzCU0TCJgX+27AKzsBOPmf/LNVAd"
    },
    {
      "ciphertext": "vault:v2:GJl6vWlY/gOF1Q6quIvRie8EG59SC811iyPuuVRfm6nba/3+/tZ+8j5YfKBOteUd0Q=="
    }
  ]
}

=== Batch decrypt results ===
[
  {
    "plaintext": "",
    "error": "invalid ciphertex

# Verifica que es posible hacer batch decoding request con cyphertext para distintas versiones

In [17]:
%%bash
# Verifica que es posible hacer batch decoding request con cyphertext para distintas versiones

# Encrypt 3 plaintexts in one batch_input request, tamper two ciphertexts, then batch decrypt

P1=$(echo -n "Alice 4111111111111111" | base64)
P2=$(echo -n "Bob 5500000000000004" | base64)
P3=$(echo -n "Carol 340000000000009" | base64)

jq -n --arg p1 "$P1" --arg p2 "$P2" --arg p3 "$P3" \
  '{batch_input: [{plaintext: $p1}, {plaintext: $p2}, {plaintext: $p3}]}' \
  > tamper_encrypt_input.json

curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data @tamper_encrypt_input.json \
     $VAULT_ADDR/v1/transit/encrypt/kms > tamper_encrypt_output_v1.json


# Rotate keys
vault write -f transit/keys/kms/rotate 

# Repeate encrypt operation with  new key verison
curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data @tamper_encrypt_input.json \
     $VAULT_ADDR/v1/transit/encrypt/kms > tamper_encrypt_output_v2.json

# Create a single batch_input request with all ciphertexts
jq -n \
  --argjson v1 "$(cat tamper_encrypt_output_v1.json)" \
  --argjson v2 "$(cat tamper_encrypt_output_v2.json)" \
  '{batch_input: [
      {ciphertext: $v1.data.batch_results[0].ciphertext},
      {ciphertext: $v2.data.batch_results[0].ciphertext}
    ]}' \
  > tamper_decrypt_input.json

echo ""
echo "=== Tampered batch_input for decrypt ==="

# Decode the batch_input request
jq . tamper_decrypt_input.json

# Decrypt the batch_input request
curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data @tamper_decrypt_input.json \
     "$VAULT_ADDR/v1/transit/decrypt/kms" | jq .


Key                       Value
---                       -----
allow_plaintext_backup    false
auto_rotate_period        720h
deletion_allowed          false
derived                   false
exportable                false
imported_key              false
keys                      map[1:1789739161 2:1789739168 3:1789739168]
latest_version            3
min_available_version     0
min_decryption_version    1
min_encryption_version    0
name                      kms
supports_decryption       true
supports_derivation       true
supports_encryption       true
supports_signing          false
type                      aes256-gcm96

=== Tampered batch_input for decrypt ===
{
  "batch_input": [
    {
      "ciphertext": "vault:v2:1kfSiQjImYrWILVCdFunVVGom3GnYP1uAEj2ZtQ+tzNRHLdP7ZsJQf9OeN4DD2a7swI="
    },
    {
      "ciphertext": "vault:v3:v1RUhyVKxObrQ87J0RDg4ZU/gcWpXYpagDhQjMC/smm9xYtyDTOu5wpNOV172/VnXgw="
    }
  ]
}
{
  "request_id": "e1aba617-0654-3856-82f2-b115c23a9bdf",
  "lease_id": "",

# Ejemplo con Python

Misma operación de cifrado y descifrado, ahora con la librería oficial [`hvac`](https://python-hvac.org/). Transit exige que el plaintext viaje en **base64**; el ciphertext que devuelve Vault tiene el formato `vault:vN:...`, donde `N` es la versión de la clave.

## Configuramos TLS Auth Method

In [18]:
%%bash
# Descargamos la CA de Vault (pki_int) del notebook 2_PKI.ipynb
# Estamos creando una dependencia circular con una CA del propio Vault - sólo como demostración
vault read -format=json pki_int/cert/ca_chain | jq -r .data.ca_chain > ${WORKDIR}/vault_ca.pem

In [19]:
%%bash
# Creamos TLS Auth Method
vault auth disable cert
vault auth enable cert

# Creamos policy transit
vault policy write transit - <<EOF
path "transit/encrypt/kms" {
    capabilities = ["update"]
}
path "transit/decrypt/kms" {
    capabilities = ["update"]
}
EOF

# Creamos certificado para la autenticación
vault write auth/cert/certs/transit display_name="transit" certificate=@${WORKDIR}/vault_ca.pem ttl=1m policies="transit" \
    token_type="batch"


Success! Disabled the auth method (if it existed) at: cert/
Success! Enabled cert auth method at: cert/
Success! Uploaded policy: transit
Success! Data written to: auth/cert/certs/transit


In [20]:
%%bash
# Creamos certificado para la autenticación
vault write -format=json pki_int/issue/example-dot-com common_name="*.test.com" ip_sans="8.8.8.9" \
    uri_sans="otrauri.example.com,masuri.example.com" ttl="1d" > ${WORKDIR}/app_cert.json

# Extraemos el certificado y la clave privada
jq -r .data.certificate ${WORKDIR}/app_cert.json > ${WORKDIR}/vault_cert.pem
jq -r .data.private_key ${WORKDIR}/app_cert.json > ${WORKDIR}/vault_key.pem


In [21]:
! pip install hvac

## Aplicación Python con TLS Auth Method

In [22]:
import base64
import hvac

WORKDIR = "/tmp/vault"

# Reuses VAULT_ADDR / VAULT_CACERT loaded from dotenv / Uses TLS Auth Method
# hvac follows requests: cert is a (cert_path, key_path) tuple. There is no key= argument;
# extra kwargs are forwarded to Adapter and raise TypeError.
client = hvac.Client(
    url=VAULT_ADDR,
    cert=(f"{WORKDIR}/vault_cert.pem", f"{WORKDIR}/vault_key.pem"),
    verify=VAULT_CACERT,
)
login = client.auth.cert.login(name="transit")
auth = login["auth"] if isinstance(login, dict) and "auth" in login else login

if not client.is_authenticated():
    raise RuntimeError("hvac client is not authenticated. Check VAULT_TOKEN / VAULT_ADDR.")

token = auth.get("client_token") or client.token
token_type = auth.get("token_type", "unknown")
prefix = token.split(".", 1)[0] if token else ""
print(f"Vault token: {token}")
print(f"Token type:  {token_type}")  # batch | service
print(f"Prefix:      {prefix}.  (hvb = batch, hvs = service)")
print(f"Renewable:   {auth.get('renewable')}")  # batch tokens are not renewable
plaintext = "Hello, Vault! (Python / hvac)"
print(f"Plaintext: {plaintext}")

# Transit requires the plaintext to be base64-encoded
encrypt_response = client.secrets.transit.encrypt_data(
    name="kms",
    plaintext=base64.b64encode(plaintext.encode("utf-8")).decode("utf-8"),
)

ciphertext = encrypt_response["data"]["ciphertext"]
print(f"Ciphertext: {ciphertext}")
print(f"Key version: {encrypt_response['data']['key_version']}")

# Decrypt: Vault returns the plaintext still base64-encoded
decrypt_response = client.secrets.transit.decrypt_data(
    name="kms",
    ciphertext=ciphertext,
)
decrypted = base64.b64decode(decrypt_response["data"]["plaintext"]).decode("utf-8")
print(f"Decrypted: {decrypted}")
assert decrypted == plaintext

# Interoperability check: decrypt the ciphertext produced earlier with curl
with open("ciphertext.txt") as f:
    old_ciphertext = f.read().strip()

old_decrypt = client.secrets.transit.decrypt_data(
    name="kms",
    ciphertext=old_ciphertext,
)
old_plaintext = base64.b64decode(old_decrypt["data"]["plaintext"]).decode("utf-8")
print(f"\nOld ciphertext (from curl): {old_ciphertext}")
print(f"Decrypted (old key version): {old_plaintext}")

Vault token: hvb.AAAAAQJdhqwB6-ofS1psl34LR8rXE2eT5cku1Kk_NvtQJS7kgDmjFjRCAvqttr8WE9GAWGsWVvOf4UnUPtGbmhSzabstsBmnfwPvf4SqJdxO-cfCVcAYGahiqejo4eEK13Oetwn8fyd2uWs2u22o3XlIgc1xdfjgGrDtW0cTnWsgdN1SBN9FTMtG0ITPvgaJq6vQzBmlQq7VlyMbkWx9ju54SeEdoK0BOSmXMG-h7PaQP-6sdjVFOgl99qGUE8vHrC5Ye9tTSDR7hfP9qKOIr2KLiGP7MZbiOdhsxZ-AKGVPBW3f2my2gZNlfUjiiS0izgLCL4hE_hI55gBvu8orgJEq4Vfg2L38Aq_vCbacIRuy1PfrjX1G8_9hfwUVmwXA62dTr-5EGnTQKzsw510FCCvfbXSmBnq8pEK0ZsRDiZcay--yHgxSBD5kxEtNv9MZ9bmGcpMqb3KgWUtWiYPxGzdg6lv3Nlg6GYltXaeOcY6YYNnGZjbqIO6MwokJbSwlsGgANYbf8BnRU2jRaKdDTdE64gCfSGy8AbMXt3W9jvgvbA
Token type:  batch
Prefix:      hvb.  (hvb = batch, hvs = service)
Renewable:   False
Plaintext: Hello, Vault! (Python / hvac)
Ciphertext: vault:v3:whJPVxEjqm7Uxjq3slgbQkiGb0y1dKTJmRLHtQTBArWwNB4SIfZpN/9nc3exETNEVvUxsbwT8AfH
Key version: 3
Decrypted: Hello, Vault! (Python / hvac)

Old ciphertext (from curl): vault:v1:FxdwhkNgzGuZqO22fv9SbreviZlirl2N8BsSHZLAy0ooR8E2OoVD96Y=
Decrypted (old key version): Hello, Vault!


# Ejemplo con Java

Misma operación de cifrado y descifrado con [Spring Cloud Vault](https://docs.spring.io/spring-cloud-vault/reference/index.html). Spring Cloud Vault auto-configura el `VaultTemplate` a partir de `spring.cloud.vault.*`; el cifrado lo hace [Spring Vault](https://docs.spring.io/spring-vault/reference/vault/vault-secret-engines.html#vault.secret.transit) (`VaultTransitOperations`). A diferencia de `hvac`, **no hace falta base64 manual**: `encrypt(key, plaintext)` y `decrypt(key, ciphertext)` lo gestionan internamente.

El proyecto se genera en `$WORKDIR/java-transit` (hace falta JDK 17+ y Maven para ejecutarlo). Usa el mismo TLS Auth Method que Python (`CERT`) para que Vault emita un **batch token**. `kv.enabled=false` evita que Spring intente montar KV como `PropertySource`; aquí solo usamos Transit.

```properties
spring.application.name=vault-transit-demo
spring.main.web-application-type=none
debug=false

spring.cloud.vault.uri=${VAULT_ADDR}
spring.cloud.vault.authentication=CERT
spring.cloud.vault.ssl.cert-auth-path=cert
spring.cloud.vault.ssl.key-store=file:client.p12
spring.cloud.vault.ssl.key-store-password=changeit
spring.cloud.vault.ssl.key-store-type=PKCS12
spring.cloud.vault.kv.enabled=false
spring.config.import=optional:vault://

transit.path=transit
transit.key=kms
```

```java
@Component
class VaultTransit {
    private final VaultOperations vault;
    private final TransitProperties props;

    VaultTransit(VaultTemplate vaultTemplate, TransitProperties props) {
        this.vault = vaultTemplate;
        this.props = props;
    }

    String encrypt(String plaintext) {
        return vault.opsForTransit(props.path()).encrypt(props.key(), plaintext);
    }

    String decrypt(String ciphertext) {
        return vault.opsForTransit(props.path()).decrypt(props.key(), ciphertext);
    }
}
```

In [23]:
from pathlib import Path
import os
import textwrap

root = Path(os.environ.get("WORKDIR", "/tmp/vault")) / "java-transit"
java_dir = root / "src/main/java/com/hashicorp/transitdemo"
res_dir = root / "src/main/resources"
java_dir.mkdir(parents=True, exist_ok=True)
res_dir.mkdir(parents=True, exist_ok=True)

truststore = root / "truststore.jks"
keystore = root / "client.p12"
vault_addr = os.environ.get("VAULT_ADDR", "https://127.0.0.1:8443")

# CERT auth requires a client keystore. Spring Cloud Vault only applies
# spring.cloud.vault.ssl.* when Apache HttpClient or OkHttp is on the classpath.
ssl_props = f"""
spring.cloud.vault.ssl.key-store=file:{keystore}
spring.cloud.vault.ssl.key-store-password=changeit
spring.cloud.vault.ssl.key-store-type=PKCS12
"""
if vault_addr.startswith("https"):
    ssl_props += f"""
spring.cloud.vault.ssl.trust-store=file:{truststore}
spring.cloud.vault.ssl.trust-store-password=changeit
spring.cloud.vault.ssl.trust-store-type=JKS
"""

(root / "pom.xml").write_text(textwrap.dedent("""\
    <?xml version=\"1.0\" encoding=\"UTF-8\"?>
    <project xmlns=\"http://maven.apache.org/POM/4.0.0\"
             xmlns:xsi=\"http://www.w3.org/2001/XMLSchema-instance\"
             xsi:schemaLocation=\"http://maven.apache.org/POM/4.0.0 https://maven.apache.org/xsd/maven-4.0.0.xsd\">
      <modelVersion>4.0.0</modelVersion>

      <parent>
        <groupId>org.springframework.boot</groupId>
        <artifactId>spring-boot-starter-parent</artifactId>
        <version>3.2.5</version>
        <relativePath/>
      </parent>

      <groupId>com.hashicorp</groupId>
      <artifactId>vault-transit-demo</artifactId>
      <version>0.0.1-SNAPSHOT</version>
      <name>vault-transit-demo</name>

      <properties>
        <java.version>17</java.version>
        <spring-cloud.version>2023.0.1</spring-cloud.version>
      </properties>

      <dependencies>
        <dependency>
          <groupId>org.springframework.boot</groupId>
          <artifactId>spring-boot-starter</artifactId>
        </dependency>
        <dependency>
          <groupId>org.springframework.cloud</groupId>
          <artifactId>spring-cloud-starter-vault-config</artifactId>
        </dependency>
        <dependency>
          <groupId>org.apache.httpcomponents.client5</groupId>
          <artifactId>httpclient5</artifactId>
        </dependency>
      </dependencies>

      <dependencyManagement>
        <dependencies>
          <dependency>
            <groupId>org.springframework.cloud</groupId>
            <artifactId>spring-cloud-dependencies</artifactId>
            <version>${spring-cloud.version}</version>
            <type>pom</type>
            <scope>import</scope>
          </dependency>
        </dependencies>
      </dependencyManagement>

      <build>
        <plugins>
          <plugin>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-maven-plugin</artifactId>
          </plugin>
        </plugins>
      </build>
    </project>
    """), encoding="utf-8")

(res_dir / "application.properties").write_text(f"""\
spring.application.name=vault-transit-demo
spring.main.web-application-type=none
debug=false

spring.cloud.vault.uri=${{VAULT_ADDR}}
spring.cloud.vault.authentication=CERT
spring.cloud.vault.ssl.cert-auth-path=cert
spring.cloud.vault.kv.enabled=false
spring.cloud.vault.config.lifecycle.enabled=false
spring.config.import=optional:vault://
{ssl_props}
transit.path=transit
transit.key=kms

logging.level.root=WARN
logging.level.org.springframework=WARN
logging.level.com.hashicorp=INFO
""", encoding="utf-8")

(java_dir / "TransitDemoApplication.java").write_text(textwrap.dedent("""\
    package com.hashicorp.transitdemo;

    import java.nio.file.Files;
    import java.nio.file.Path;

    import org.springframework.boot.CommandLineRunner;
    import org.springframework.boot.SpringApplication;
    import org.springframework.boot.autoconfigure.SpringBootApplication;
    import org.springframework.boot.context.properties.ConfigurationProperties;
    import org.springframework.boot.context.properties.EnableConfigurationProperties;
    import org.springframework.stereotype.Component;
    import org.springframework.vault.authentication.SessionManager;
    import org.springframework.vault.core.VaultOperations;
    import org.springframework.vault.core.VaultTemplate;

    @SpringBootApplication
    @EnableConfigurationProperties(TransitProperties.class)
    public class TransitDemoApplication {

        public static void main(String[] args) {
            SpringApplication.run(TransitDemoApplication.class, args);
        }
    }

    @ConfigurationProperties(prefix = \"transit\")
    record TransitProperties(String path, String key) {
    }

    @Component
    class VaultTransit {
        private final VaultOperations vault;
        private final TransitProperties props;

        VaultTransit(VaultTemplate vaultTemplate, TransitProperties props) {
            this.vault = vaultTemplate;
            this.props = props;
        }

        String encrypt(String plaintext) {
            return vault.opsForTransit(props.path()).encrypt(props.key(), plaintext);
        }

        String decrypt(String ciphertext) {
            return vault.opsForTransit(props.path()).decrypt(props.key(), ciphertext);
        }
    }

    @Component
    class TransitDemo implements CommandLineRunner {
        private final VaultTransit transit;
        private final SessionManager sessionManager;
        private final VaultTemplate vault;

        TransitDemo(VaultTransit transit, SessionManager sessionManager, VaultTemplate vault) {
            this.transit = transit;
            this.sessionManager = sessionManager;
            this.vault = vault;
        }

        @Override
        public void run(String... args) throws Exception {
            printVaultToken();

            String plaintext = \"Hello, Vault! (Java / Spring Cloud Vault)\";
            System.out.println(\"Plaintext: \" + plaintext);

            String ciphertext = transit.encrypt(plaintext);
            System.out.println(\"Ciphertext: \" + ciphertext);

            String decrypted = transit.decrypt(ciphertext);
            System.out.println(\"Decrypted: \" + decrypted);
            if (!plaintext.equals(decrypted)) {
                throw new IllegalStateException(\"Decrypted text does not match plaintext\");
            }

            Path oldCipherFile = Path.of(System.getenv().getOrDefault(
                    \"CIPHERTEXT_FILE\", \"ciphertext.txt\"));
            if (Files.exists(oldCipherFile)) {
                String oldCiphertext = Files.readString(oldCipherFile).trim();
                System.out.println();
                System.out.println(\"Old ciphertext (from curl): \" + oldCiphertext);
                System.out.println(\"Decrypted (old key version): \"
                        + transit.decrypt(oldCiphertext));
            }
        }

        private void printVaultToken() {
            String token = sessionManager.getSessionToken().getToken();
            String prefix = token.contains(\".\") ? token.substring(0, token.indexOf('.')) : \"\";
            System.out.println(\"Vault token: \" + token);
            System.out.println(\"Prefix:      \" + prefix + \".  (hvb = batch, hvs = service)\");
            try {
                var lookup = vault.read(\"auth/token/lookup-self\");
                var data = lookup.getData();
                System.out.println(\"Token type:  \" + data.get(\"type\"));
                System.out.println(\"Renewable:   \" + data.get(\"renewable\"));
            } catch (Exception ex) {
                System.out.println(\"Token lookup-self failed: \" + ex.getMessage());
            }
            System.out.println();
        }
    }
    """), encoding="utf-8")

print(f"Proyecto Spring Cloud Vault escrito en {root}")

Proyecto Spring Cloud Vault escrito en /tmp/vault/java-transit


In [24]:
%%bash
set -euo pipefail

ROOT="${WORKDIR:-/tmp/vault}/java-transit"

# Jupyter kernels do not source ~/.zshrc. Homebrew OpenJDK is keg-only, so
# /usr/bin/java is Apple's stub and "java -version" fails until JAVA_HOME is set.
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"

if [ -z "${JAVA_HOME:-}" ] || [ ! -x "${JAVA_HOME}/bin/java" ]; then
  for d in \
    /opt/homebrew/opt/openjdk@17 \
    /opt/homebrew/opt/openjdk@21 \
    /opt/homebrew/opt/openjdk \
    /usr/local/opt/openjdk@17 \
    /usr/local/opt/openjdk@21 \
    /usr/local/opt/openjdk
  do
    home="$d/libexec/openjdk.jdk/Contents/Home"
    if [ -x "$home/bin/java" ]; then
      export JAVA_HOME="$home"
      break
    fi
  done
fi

if [ -n "${JAVA_HOME:-}" ]; then
  export PATH="$JAVA_HOME/bin:$PATH"
fi

if ! command -v java >/dev/null 2>&1 || ! java -version >/dev/null 2>&1; then
  echo "JDK 17+ not found. Install a JDK and Maven to run this example."
  echo "Project generated at: $ROOT"
  exit 0
fi

if ! command -v mvn >/dev/null 2>&1; then
  echo "Maven not found. Install Maven and run:"
  echo "  mvn -f $ROOT/pom.xml spring-boot:run"
  echo "Project generated at: $ROOT"
  exit 0
fi

if [ -n "${VAULT_CACERT:-}" ] && [ -f "$VAULT_CACERT" ]; then
  rm -f "$ROOT/truststore.jks"
  keytool -importcert -noprompt -alias vault \
    -file "$VAULT_CACERT" \
    -keystore "$ROOT/truststore.jks" \
    -storepass changeit
  echo "Imported $VAULT_CACERT into $ROOT/truststore.jks"
fi

# Same client cert as the Python TLS login, packaged as PKCS12 for Spring CERT auth
CERT_DIR="${WORKDIR:-/tmp/vault}"
openssl pkcs12 -export \
  -in "$CERT_DIR/vault_cert.pem" \
  -inkey "$CERT_DIR/vault_key.pem" \
  -out "$ROOT/client.p12" \
  -name transit \
  -passout pass:changeit
echo "Created $ROOT/client.p12 for Spring Cloud Vault CERT authentication"

export CIPHERTEXT_FILE="$(pwd)/ciphertext.txt"
# Spring Boot binds env DEBUG to debug=true and dumps CONDITIONS EVALUATION REPORT.
unset DEBUG
mvn -f "$ROOT/pom.xml" -q spring-boot:run -Dspring-boot.run.arguments=--debug=false


Certificate was added to keystore


Imported /tmp/vault/vault.ca into /tmp/vault/java-transit/truststore.jks
Created /tmp/vault/java-transit/client.p12 for Spring Cloud Vault CERT authentication

  .   ____          _            __ _ _
 /\\ / ___'_ __ _ _(_)_ __  __ _ \ \ \ \
( ( )\___ | '_ | '_| | '_ \/ _` | \ \ \ \
 \\/  ___)| |_)| | | | | || (_| |  ) ) ) )
  '  |____| .__|_| |_|_| |_\__, | / / / /
 =========|_|==============|___/=/_/_/_/
 :: Spring Boot ::                (v3.2.5)

2026-09-18T15:46:13.903+02:00  INFO 5355 --- [vault-transit-demo] [           main] c.h.transitdemo.TransitDemoApplication   : Starting TransitDemoApplication using Java 17.0.20.1 with PID 5355 (/private/tmp/vault/java-transit/target/classes started by jose in /private/tmp/vault/java-transit)
2026-09-18T15:46:13.905+02:00  INFO 5355 --- [vault-transit-demo] [           main] c.h.transitdemo.TransitDemoApplication   : No active profile set, falling back to 1 default profile: "default"
2026-09-18T15:46:14.308+02:00  INFO 5355 --- [vault-transi

# Long-lived apps in Kubernetes

The cert role issues a **batch token with TTL=1m**. Batch tokens cannot be renewed, so each app must **log in again** with the client certificate when the token is about to expire.

- Python (`hvac`): encrypt/decrypt every 20s; re-login when remaining TTL < 10s (and on 403)
- Java (Spring Cloud Vault): same loop via `@Scheduled`; session lifecycle + retry on Vault errors

Images are built into minikube (`transit-python:demo`, `transit-java:demo`) and run in namespace `transit-apps`. Watch logs: a new `LOGIN` / `NEW TOKEN` line after ~1 minute means rotation worked.

In [ ]:
%%bash
set -euo pipefail

REPO="$(pwd)"
WORKDIR="${WORKDIR:-/tmp/vault}"
NS=transit-apps

for f in vault.ca vault_cert.pem vault_key.pem; do
  if [ ! -f "$WORKDIR/$f" ]; then
    echo "Missing $WORKDIR/$f — run the TLS Auth cells first."
    exit 1
  fi
done

echo "Building images inside minikube..."
minikube image -p workshop build -t transit-python:demo "$REPO/transit-apps/python"
minikube image -p workshop build -t transit-java:demo "$REPO/transit-apps/java"

kubectl create namespace "$NS" --dry-run=client -o yaml | kubectl apply -f -

kubectl -n "$NS" create secret generic vault-tls \
  --from-file=ca.pem="$WORKDIR/vault.ca" \
  --from-file=client.pem="$WORKDIR/vault_cert.pem" \
  --from-file=client-key.pem="$WORKDIR/vault_key.pem" \
  --dry-run=client -o yaml | kubectl apply -f -

kubectl apply -f "$REPO/transit-apps/k8s.yaml"
kubectl -n "$NS" rollout restart deploy/transit-python deploy/transit-java || true
kubectl -n "$NS" rollout status deploy/transit-python --timeout=180s
kubectl -n "$NS" rollout status deploy/transit-java --timeout=300s

echo
kubectl -n "$NS" get pods -o wide
echo
echo "Follow token rotation (TTL=1m):"
echo "  kubectl -n $NS logs -f deploy/transit-python"
echo "  kubectl -n $NS logs -f deploy/transit-java"


Building images inside minikube...


#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 256B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.12-slim
#2 DONE 1.3s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [internal] load build context
#4 transferring context: 3.28kB done
#4 DONE 0.0s

#5 [1/4] FROM docker.io/library/python:3.12-slim@sha256:78387bc3881b8273120a12ebe6c1ab22b018ccc2c9adf565ae1ac9b536e184ea
#5 resolve docker.io/library/python:3.12-slim@sha256:78387bc3881b8273120a12ebe6c1ab22b018ccc2c9adf565ae1ac9b536e184ea done
#5 sha256:8aff2d3a9af8ed70ae2aa065663f6a7b99d3cd41564528e8d5f039ec0faae595 0B / 12.05MB 0.1s
#5 sha256:78387bc3881b8273120a12ebe6c1ab22b018ccc2c9adf565ae1ac9b536e184ea 10.37kB / 10.37kB done
#5 sha256:3949e4271b0a3ff82afac7306764c313dcc8edeeb89c0376a3c2ac6007c66b1d 1.75kB / 1.75kB done
#5 sha256:4f8d1afed6d58037c680221ca6dd9fb4737b7ecfa7d4809ca809f

In [ ]:
%%bash
# Wait past the 1m token TTL so logs should show a second LOGIN / NEW TOKEN
set -euo pipefail
NS=transit-apps

echo "Waiting 90s for batch token TTL (1m) to expire and apps to re-login..."
sleep 90

echo
echo "===== Python ====="
kubectl -n "$NS" logs deploy/transit-python --tail=40

echo
echo "===== Java ====="
kubectl -n "$NS" logs deploy/transit-java --tail=40


# Notas sobre hvac y Spring Cloud Vault

Tanto **hvac** como **Spring Cloud Vault** pueden usar un **batch token** de Vault (`hvb.…`). Transit encrypt/decrypt solo envían `X-Vault-Token`; Vault no exige un service token para esas APIs.

Un batch token **no se puede renovar** (`renewable=false`). La aplicación debe **volver a autenticarse** (certificado TLS en esta demo) para obtener un token nuevo. El TTL no está en la cadena opaca `hvb.`; viene de la respuesta de login (`auth.lease_duration`) o de `auth/token/lookup-self` (`ttl` / `expire_time`). Spring mapea el TTL del login a `LoginToken.getLeaseDuration()`.

Lo que cambia es el ciclo de vida de la sesión:

- **Python / hvac** no gestiona el TTL. Tras el login con certificado guarda `lease_duration` y, cuando el TTL restante es &lt; 10s, **se reautentica con el certificado antes de que expire**. En el camino feliz no hay 403.
- **Spring Cloud Vault** guarda el token en `LifecycleAwareSessionManager`. El refresh (`renew-self`) solo se ejecuta si el token es **renewable**, así que un batch token se usa hasta que Vault responde **403 invalid token**. Esta demo entonces llama a `destroy()` + `getSessionToken()`, que es otro login con certificado.

El mismo patrón proactivo que en Python es posible en Java: leer `LoginToken.getLeaseDuration()`, volver a hacer login `skew` segundos antes de que expire, y mantener el handler 403 como fallback. `spring.cloud.vault.session.lifecycle.refresh-before-expiry` no ayuda aquí: ese camino es **renovación** de token, no **re-login** con certificado.

# Vault Benchmark

[`vault-benchmark`](https://github.com/hashicorp/vault-benchmark) genera carga contra Transit: **encrypt 95%** / **decrypt 5%**, clave AES-256-GCM (`orders`). Decrypt usa el ciphertext que el tool siembra en el setup (un item).

HCL en `vault-benchmark/`. Cada run imprime el informe, lo guarda en `vault-benchmark/results-<label>.txt` y extrapola a **items** (requests × tamaño de `batch_input` × success).

1. **Unitario** — sin `batch_input`, **10 workers**. Una cifra/descifra por request.
2. **Batch 1000** — mismo cardinal que `batch_input.json`.
3. **Batch ~9000** — por debajo de [`max_json_array_element_count`](https://developer.hashicorp.com/vault/docs/configuration/listener/tcp#max_json_array_element_count) (default 10 000).


In [32]:
%%bash
set -euo pipefail
WORKDIR="${WORKDIR:-/tmp/vault}"
mkdir -p "$WORKDIR"

VB_VER=0.3.0
OS=$(uname -s | tr '[:upper:]' '[:lower:]')
ARCH=$(uname -m)
case "$ARCH" in
  x86_64) ARCH=amd64 ;;
  aarch64|arm64) ARCH=arm64 ;;
esac

ZIP="vault-benchmark_${VB_VER}_${OS}_${ARCH}.zip"
SUMS="vault-benchmark_${VB_VER}_SHA256SUMS"
BASE="https://releases.hashicorp.com/vault-benchmark/${VB_VER}"

echo "Downloading ${BASE}/${ZIP}"
curl -fsSL -o "$WORKDIR/$ZIP" "$BASE/$ZIP"
curl -fsSL -o "$WORKDIR/$SUMS" "$BASE/$SUMS"
expect=$(awk -v f="$ZIP" '$2==f {print $1}' "$WORKDIR/$SUMS")
actual=$(shasum -a 256 "$WORKDIR/$ZIP" | awk '{print $1}')
if [ -z "$expect" ] || [ "$expect" != "$actual" ]; then
  echo "SHA256 mismatch for $ZIP" >&2
  echo "expected=$expect actual=$actual" >&2
  exit 1
fi
echo "SHA256 OK ($actual)"
unzip -o "$WORKDIR/$ZIP" -d "$WORKDIR"

# Released 0.3.0 panics on encrypt.batch_input: gohcl cannot decode []interface{}.
# Rebuild the same tag with BatchInput as []map[string]string.
SRC="$WORKDIR/vault-benchmark-${VB_VER}"
rm -rf "$SRC"
curl -fsSL -o "$WORKDIR/vault-benchmark-${VB_VER}.tar.gz" \
  "https://github.com/hashicorp/vault-benchmark/archive/refs/tags/v${VB_VER}.tar.gz"
tar -xzf "$WORKDIR/vault-benchmark-${VB_VER}.tar.gz" -C "$WORKDIR"
perl -pi -e 's/BatchInput\s+\[\]interface\{\}/BatchInput []map[string]string/g' \
  "$SRC/benchmarktests/target_secret_transit.go"
( cd "$SRC" && go build -o "$WORKDIR/vault-benchmark" . )
chmod +x "$WORKDIR/vault-benchmark"
"$WORKDIR/vault-benchmark" -h | head -6


SHA256 OK (f169153805702f6642e576c6ce52e139ddcd71e03b60f63896f58ec4bb2be2fc)
Archive:  /tmp/vault/vault-benchmark_0.3.0_darwin_arm64.zip
  inflating: /tmp/vault/LICENSE.txt  
  inflating: /tmp/vault/vault-benchmark  
Usage: vault-benchmark <command> [args]

Command list:
    run         Run vault-benchmark test(s)
    review      Review previous test results


## 1. Operaciones unitarias (10 workers)

Sin `batch_input`: cada request cifra o descifra **un** PAN. Más workers (10) para saturar con payloads pequeños.


In [49]:
%%bash
set -euo pipefail
set -a
# shellcheck disable=SC1091
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-vb.sh ./vault-benchmark/vb-transit-unit.hcl unit


=== vault-benchmark unit  (encrypt x1 / decrypt x1 per request) ===
2026-09-18T18:47:53.884+0200 [INFO]  vault-benchmark: setting up targets
2026-09-18T18:47:53.964+0200 [INFO]  vault-benchmark: starting benchmarks: duration=1m0s
2026-09-18T18:48:54.001+0200 [INFO]  vault-benchmark: cleaning up targets
2026-09-18T18:48:54.027+0200 [INFO]  vault-benchmark: benchmark complete
Target: https://127.0.0.1:8443
op            count  rate         throughput   mean        95th%        99th%        successRatio
unit_decrypt  3549   59.165289    59.162649    8.614641ms  43.029764ms  44.053602ms  100.00%
unit_encrypt  68151  1135.193011  1135.173140  8.352379ms  43.017838ms  43.982194ms  100.00%


Saved /Users/jose/Library/CloudStorage/GoogleDrive-jose.maria.merchan@gmail.com/My Drive/Demo/Vault_on_Kubernetes_Webinar/vault-benchmark/results-unit.txt

=== Items (successful requests × items/request) ===
  unit_decrypt      req=3549  ok=100.00%  items=3549  items/s=59  (x1 / request)
  unit_encrypt   

## 2. Batch 1000 entradas

Misma mezcla 95/5, `workers = 2`. Cada encrypt request cifra **1000** PAN.


In [50]:
%%bash
set -euo pipefail
set -a
# shellcheck disable=SC1091
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-vb.sh ./vault-benchmark/vb-transit-1000.hcl batch-1000


=== vault-benchmark batch-1000  (encrypt x1000 / decrypt x1 per request) ===
2026-09-18T18:50:23.375+0200 [INFO]  vault-benchmark: setting up targets
2026-09-18T18:50:23.451+0200 [INFO]  vault-benchmark: starting benchmarks: duration=1m0s
2026-09-18T18:51:23.545+0200 [INFO]  vault-benchmark: cleaning up targets
2026-09-18T18:51:23.603+0200 [INFO]  vault-benchmark: benchmark complete
Target: https://127.0.0.1:8443
op             count  rate        throughput  mean         95th%        99th%        successRatio
batch_decrypt  333    5.572571    5.571426    18.541542ms  42.722084ms  43.498315ms  100.00%
batch_encrypt  6113   101.812461  101.725634  18.637993ms  33.360103ms  51.029813ms  100.00%


Saved /Users/jose/Library/CloudStorage/GoogleDrive-jose.maria.merchan@gmail.com/My Drive/Demo/Vault_on_Kubernetes_Webinar/vault-benchmark/results-batch-1000.txt

=== Items (successful requests × items/request) ===
  batch_decrypt     req=333  ok=100.00%  items=333  items/s=6  (x1 / request)
  bat

## 3. Batch ~9000 entradas

Misma mezcla 95/5. Por debajo del límite de 10 000 elementos JSON del listener.


In [51]:
%%bash
set -euo pipefail
set -a
# shellcheck disable=SC1091
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-vb.sh ./vault-benchmark/vb-transit-9000.hcl batch-9000


=== vault-benchmark batch-9000  (encrypt x8982 / decrypt x1 per request) ===
2026-09-18T18:51:41.311+0200 [INFO]  vault-benchmark: setting up targets
2026-09-18T18:51:41.347+0200 [INFO]  vault-benchmark: starting benchmarks: duration=1m0s
2026-09-18T18:52:41.570+0200 [INFO]  vault-benchmark: cleaning up targets
2026-09-18T18:52:41.626+0200 [INFO]  vault-benchmark: benchmark complete
Target: https://127.0.0.1:8443
op             count  rate       throughput  mean          95th%         99th%         successRatio
batch_decrypt  39     0.728894   0.728326    33.425925ms   47.482283ms   55.321417ms   100.00%
batch_encrypt  818    13.612151  13.582947   145.559133ms  183.728764ms  202.683714ms  100.00%


Saved /Users/jose/Library/CloudStorage/GoogleDrive-jose.maria.merchan@gmail.com/My Drive/Demo/Vault_on_Kubernetes_Webinar/vault-benchmark/results-batch-9000.txt

=== Items (successful requests × items/request) ===
  batch_decrypt     req=39  ok=100.00%  items=39  items/s=1  (x1 / request)
 

# Clean Up

Elimina lo creado en este notebook: namespace `transit-apps` (pods, deployments, secret TLS), imágenes en minikube, Transit, cert auth, policy y ficheros temporales. No toca el cluster de Vault ni el PKI del notebook 2.